# MMS–bow-shock magnetic connection

This notebook mirrors `mms_bow_shock_connection.py`: it extracts a bow shock from a BATSRUS Tecplot file, reads the effective time, averaged `Bavg`, and MMS location from a PARAM file created by `create_swmf_input.py`, finds the closest shock intersection, and displays 2D and 3D views.

The standard installation includes MMS loading and plotting support. Install the notebook tools with `pip install -e ".[notebook]"`, then launch from the repository root with:

```bash
jupyter lab examples/shock_connection.ipynb
```

The Tecplot sample is large. Keep the notebook unexecuted when committing it.

## Parameters

Edit the simulation and PARAM paths, extraction resolution, and smoothing controls before running the analysis. The PARAM file must be created by `create_swmf_input.py` so its timestamp, averaged magnetic field, and GSM MMS location are available.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from shocklink.bowshock import (
    calc_bow_shock_normals,
    extract_shockfit_range,
    fit_bow_shock,
    get_bow_shock_surface,
    smooth_bow_shock_surface,
)
from shocklink.connectivity import (
    analyze_shock_connection,
    plot_shock_angle_contour,
    plot_shock_connection_3d,
)
from shocklink.dataset import calc_velocity_divergence
from shocklink.io import load_simulation
from shocklink.swmf import read_mms_param_file

pv.set_jupyter_backend("static")

DATA_PATH = Path("../data/3d.dat")
PARAM_PATH = Path("../results/PARAM_20181219_194600.in")
SURFACE_AXIS = np.linspace(-30.0, 30.0, 241)
X_RESOLUTION = 512
CHUNK_SIZE = 1024
SMOOTHING_SIGMA = 5.0
SHOCKFIT_RANGE = (-5.0, 5.0)

## Extract and smooth the bow shock

The extracted `surface_x` array is indexed as `[Y, Z]`. Missing columns remain `NaN`; the normal calculation fills supported gaps for differentiation, while the connection plots keep unsupported shock cells masked.

In [ ]:
grid = load_simulation(DATA_PATH)
calc_velocity_divergence(grid)
fit = fit_bow_shock(grid)
shock_region = extract_shockfit_range(
    grid,
    lower=SHOCKFIT_RANGE[0],
    upper=SHOCKFIT_RANGE[1],
)

surface_x_raw = get_bow_shock_surface(
    shock_region,
    x_resolution=X_RESOLUTION,
    chunk_size=CHUNK_SIZE,
    refine_minimum=True,
)
surface_x = smooth_bow_shock_surface(surface_x_raw, sigma=SMOOTHING_SIGMA)
normals = calc_bow_shock_normals(
    surface_x, y=SURFACE_AXIS, z=SURFACE_AXIS
)

print(f"Fitted nose X: {fit.loc0[0]:.3f} R_E")
print(f"Finite surface samples: {np.isfinite(surface_x).sum():,}/{surface_x.size:,}")

## Load MMS and calculate the connection

The PARAM file supplies the effective time, interval-averaged GSM magnetic field, and MMS location. The normalized `Bavg` vector defines an infinite straight line in both directions; if it crosses the shock more than once, the result selects the crossing closest to MMS.

In [ ]:
param_values = read_mms_param_file(PARAM_PATH)
mms_position = np.array(
    [param_values.location.x, param_values.location.y, param_values.location.z],
    dtype=float,
)
bavg = np.array(
    param_values.magnetic_field,
    dtype=float,
)

connection = analyze_shock_connection(
    surface_x,
    normals,
    y=SURFACE_AXIS,
    z=SURFACE_AXIS,
    mms_position=mms_position,
    bavg=bavg,
)
hit = connection.selected_intersection
print(f"MMS PARAM time: {param_values.time.isoformat()} (GSM)")
print(f"MMS position [R_E]: {mms_position}")
print(f"Bavg [nT]: {bavg}")
print(f"Intersection [R_E]: {hit.point}")
print(f"Signed line parameter: {hit.line_parameter:.6g}")
print(f"Distance from MMS: {hit.distance:.6g} R_E")
print(f"theta_Bn: {hit.theta_bn_deg:.3f} deg")

## Plot the connection

The 2D view shows the acute `theta_Bn` field over the extracted Y–Z shock and marks the selected intersection. The 3D view adds Earth, the colored shock, MMS, the straight field line, the Bavg arrow, and the intersection.

In [ ]:
figure, axes = plot_shock_angle_contour(connection)

In [ ]:
plotter = plot_shock_connection_3d(connection, show=False)
plotter.show(jupyter_backend="static")